## TUGAS MANDIRI - Debi Aprilia - 2320506038

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum UTS dimulai** (Pertemuan 8 adalah UTS, tidak ada praktikum — namun tugas ini tetap wajib dikumpulkan sesuai tenggat agar tidak menumpuk menjelang pertemuan berikutnya).
> **Sifat tugas:** individu.

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh **"Kopi Nusantara"**, jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.



### Membuat SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/01 12:28:13 WARN Utils: Your hostname, deb-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/01 12:28:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 12:28:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data dan mengunggah data transaksi ke HDFS.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


### Pipeline Analisis

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

In [4]:
#Membaca data cabang (JSON)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")

#Membaca data menu (CSV)
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

#Membaca data transaksi (CSV)
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)

#Verifikasi jumlah baris
print("Cabang:", df_cabang.count(), "| Menu:", df_menu.count(), "| Transaksi:", df_transaksi.count())

Cabang: 3 | Menu: 15 | Transaksi: 4000


**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

In [5]:
#Menggabungkan ketiga tabel
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner").join(df_menu, on="menu_id", how="inner")

#Menambahkan kolom total_penjualan
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

#Menampilkan hasil 
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.


In [6]:
#Agregasi total penjualan per kombinasi cabang dan menu
df_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(spark_sum("total_penjualan").alias("total_penjualan"))

#Membuat spesifikasi window
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

#Memberi nomor urut dan filter top-2
top2_menu = df_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)).filter(col("peringkat") <= 2)

#Menamp[ilkan hasil
top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.


In [7]:
#Mendaftarkan DataFrame ke Temporary View
df_gabungan.createOrReplaceTempView("v_kopi")

#Kueri SQL
sql_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM v_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')

#Menampilkan hasil
sql_barista.show()

[Stage 32:>                                                         (0 + 1) / 1]

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [8]:
#Menyimpan ke HDFS dengan partisi
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet("hdfs://localhost:9000/user/deb/tugas7/processed/laporan")

#Membaca kembali untuk verifikasi dengan menghitung jumlah baris
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/deb/tugas7/processed/laporan")
print("Verifikasi total baris:", df_verifikasi.count())

Verifikasi total baris: 4000


**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [9]:
#Membuat tabel ringkasan
ringkasan_cabang = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista") \
    .agg(
        spark_sum("total_penjualan").alias("total_penjualan"),
        count("trx_id").alias("jumlah_transaksi"),
        avg("total_penjualan").alias("rata_rata_transaksi")
    ) \
    .orderBy(col("total_penjualan").desc())

#Menampilkan tabel
ringkasan_cabang.show()

[Stage 44:>                                                         (0 + 1) / 1]

+--------------------+----------+--------------+---------------+----------------+-------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|   64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|  66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|  64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------+



**Rekomendasi** : Berdasarkan tabel ringkasan di atas, cabang dengan kinerja terbaik secara finansial adalah cabang Kopi Nusantara Semarang dengan menghasilkan total penjualan sebesar 90.542.000. Hal ini didorong oleh tingginya jumlah transaksi dan popularitas menu utama di cabang tersebut. Sedangkan cabang dengan kinerja terlemah adalah cabang Kopi Nusantara Yogyakarta dengan menghasilkan total penjualan sebesar 84.645.000. Berdasarkan hasil analisis Window Function pada bagian C, menu yang menjadi favorit pelanggan di cabang ini adalah Menu-12 dan Menu-15. Oleh karena itu, untuk meningkatkan performa cabang ini dapat dilakukan dengan melakukan promosi lebih gencar pada menu unggulan tersebut dengan paket bundling bersama menu makanan ringan, memberi diskon di waktu - waktu tertentu, serta meningkatkan standar pelayanan di bawah koordinasi barista. Peningkatan pemasaran pada produk yang sudah terbukti banyak peminatnya akan menaikkan rata - rata transaksi per pelanggan secara signifikan.  

## Menutup SparkSession

In [10]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
